# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the dataset metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields, and their `@id`s.

We will list all record sets and their field `@id`s to guide further extraction and analysis.

In [ ]:
# List all record sets and relevant fields with their @id
print("Available Record Sets in Dataset:\n")
for record_set in dataset.metadata.record_sets:
    print(f"- Record Set @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', None)}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Available Fields and Columns (by @id):")
        for field in record_set.fields:
            # For each field, print its @id, name, and mapping to column (if present)
            col_id = getattr(field, 'column', None)
            if col_id is not None and hasattr(col_id, 'id'):
                col_id = col_id.id
            print(f"    - Field @id: {field.id}, Name: {getattr(field, 'name', None)}, Maps to Column: {col_id}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll load each record set by its `@id` as shown.

In [ ]:
# Extract data from all available record sets
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded Record Set: {record_set_id} ({len(records)} records)")

# Display columns from the primary record set (choose the first one)
primary_record_set_id = record_sets_ids[0] if record_sets_ids else None
if primary_record_set_id is not None:
    print(f"Columns in primary record set ({primary_record_set_id}):")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

We will:
- Select a numeric field (for example, 'MW [kDa]' or 'Coverage [%]' if present)
- Filter records above a threshold value
- Normalize the field
- Optionally group by another field if it exists

In [ ]:
# Pick appropriate fields for demo: try common mass-spec fields
# Adjust the following variables as per the actual columns present
import numpy as np

# Choose the main record set for EDA
record_set_id = primary_record_set_id
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()

# Try numeric fields in the dataset, e.g., 'MW [kDa]' (Molecular Weight), 'Coverage [%]', or 'Spectral Counts'
# You can list available fields/columns above to adjust this selection
candidate_fields = ['MW [kDa]', 'Coverage [%]', 'Unique Peptides', 'Total Peptides']
numeric_field = None
for fld in candidate_fields:
    if fld in df.columns:
        numeric_field = fld
        break
if numeric_field:
    print(f"Chosen numeric field for analysis: {numeric_field}")
else:
    print("No obvious numeric fields found, please check actual column names.")
    numeric_field = df.select_dtypes(include=[np.number]).columns[0] if len(df.select_dtypes(include=[np.number]).columns) > 0 else None

if numeric_field:
    # Convert the column to numeric if needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75) # upper quartile as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field if present, e.g., 'Description' or 'Accession'
    group_field = None
    for fld in ['Description', 'Accession', 'Sample']:
        if fld in df.columns:
            group_field = fld
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Example: Plot histogram of the numeric field and a bar plot of means by grouping variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field and not df.empty:
    # Plot histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Plot barplot of means by grouping field if applicable
    if 'grouped_df' in locals() and group_field and not grouped_df.empty:
        plt.figure(figsize=(12, 5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded all record sets from the FAIR² mass spectrometry dataset using their `@id`s.
- We identified primary numeric fields in the main table/record set, performed basic filtering, normalization, and grouped analysis.
- Visualization provided insights into the distribution and group-wise differences for the selected quantitative variable.
- For more advanced analysis, further biological and domain knowledge can be used to interpret specific proteins, peptides, or experimental conditions.

**Notebook complete.**